# Day 16 - Image Variations & Video Generation

## RetailMind AI Increment

Day 16 extends the Day 15 creative module from text-to-image concepts to **image-to-image product variations**. A real catalog image becomes the starting point, controlled Stable Diffusion strength/guidance sweeps produce campaign alternatives, and curated assets become an ordered frame sequence for a short promotional clip.

### Business objective
The goal is to scale creative output without completely losing the identity of the original retail product.

## Dataset and runtime

**Input dataset:** Fashion Product Images Small or another approved product catalog image. Unlike Day 15, Day 16 begins from an actual image rather than only product metadata.

Expected local file:

```text
data/day16_catalog_init.jpg
```

Generation uses the Stable Diffusion v1.5 img2img pipeline and requires a CUDA GPU. The pretrained model must be accessible from Hugging Face or already cached.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day16_image_variations_video as day16

print("Model:", day16.MODEL_ID)
print("Strength sweep:", day16.STRENGTHS)
print("Guidance sweep:", day16.GUIDANCE_SCALES)
print("Seeds:", day16.SEEDS)

## 1. Load the catalog image

The initial product image is resized to 512×512 RGB. For real campaign work, use an approved source image with clear product boundaries and sufficient resolution.

In [ ]:
# Run when the catalog image exists.
# init_image = day16.load_init_image(repo_root / day16.INIT_IMAGE)
# display(init_image)

## 2. Understand img2img strength

`strength` is the most important Day 16 control:

- **0.25:** high fidelity, small reinterpretation.
- **0.45:** balanced product preservation and creative change.
- **0.65:** stronger reinterpretation, higher risk of changing SKU details.

The correct value depends on whether the asset is a faithful catalog variation or a looser campaign concept.

In [ ]:
for strength in day16.STRENGTHS:
    print(f"strength={strength:.2f}")

## 3. Guidance-scale sweep

Guidance controls how strongly the output follows the text prompt. Testing strength and guidance together is more useful than selecting either in isolation. The project generates a 3×3 grid of combinations and records every setting in JSON.

In [ ]:
for strength in day16.STRENGTHS:
    for guidance in day16.GUIDANCE_SCALES:
        print(f"strength={strength:.2f}, guidance={guidance:.1f}")

## 4. Generate controlled product variations

A fixed seed makes a candidate reproducible. The negative prompt explicitly discourages changing product shape, duplicating the product, adding text, and creating distorted geometry.

In [ ]:
# GPU execution:
# pipe = day16.load_pipeline()
# variation = day16.generate_variation(
#     pipe, init_image, strength=0.45, guidance_scale=7.5, seed=42
# )
# display(variation)

## 5. Build and review the variation grid

The generated manifest records image path, strength, guidance, seed, model, positive prompt, negative prompt, approval state and review notes. Marketing should select assets based on identity preservation, campaign usefulness, realism and absence of artifacts.

In [ ]:
# manifest = day16.variation_grid(pipe, init_image, repo_root / day16.OUTPUT_DIR)
# print("Generated candidates:", len(manifest))
# manifest[:2]

## 6. Curate campaign-ready assets

A strong creative workflow does not automatically publish every generated image. Review each candidate for:

1. exact product shape and component count;
2. recognizable category and material;
3. approved color/style range;
4. no hallucinated controls, logos or text;
5. composition suitable for the intended placement;
6. no misleading functional claims.

Only approved filenames should be marked `approved=true` in the manifest.

In [ ]:
# Example after visual review:
# approved = [manifest[0]["file"], manifest[4]["file"]]
# manifest = day16.curate_manifest(manifest, approved, repo_root / day16.MANIFEST_PATH)

## 7. Create a short promo-frame sequence

Two approved still images can be transformed into an ordered cross-fade sequence. The helper writes 24 PNG frames. This keeps the creative generation inside the Diffusers workflow while leaving final MP4/GIF container encoding to whatever approved video encoder exists in the hosted environment.

In [ ]:
# Example after selecting two approved images:
# from PIL import Image
# first = Image.open(approved[0])
# second = Image.open(approved[1])
# frames = day16.interpolate_frames(first, second, frame_count=24,
#                                    output_dir=repo_root / day16.FRAME_DIR)
# print("Frames created:", len(frames), frames[0], "...", frames[-1])

## 8. Creative-module integration

Day 15 generated text-to-image campaign concepts. Day 16 adds product-conditioned img2img variations and a promo-frame sequence. The combined creative module now has three levels:

```text
Product metadata -> text-to-image concepts
Catalog image     -> identity-preserving img2img variations
Approved stills   -> ordered promo frames / short clip
```

### Important limitation
Cross-faded frames create simple visual motion, not true physics-aware video generation. A future dedicated video-diffusion pipeline can generate temporal motion when the approved environment provides an appropriate pretrained model and GPU capacity.